In [1]:
import pandas as pd
import numpy as np
import joblib
import json

from pathlib import Path
from datetime import datetime

import tensorflow as tf

import logging
import os
from datetime import datetime

os.makedirs("logs", exist_ok=True)
logger = logging.getLogger("Model_DL_Customer")
logger.setLevel(logging.INFO)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
if not logger.handlers:

    # Log file with timestamp
    log_file = f"logs/model_customer_{datetime.now():%Y%m%d}.log"

    # File handler
    file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
    file_handler.setLevel(logging.INFO)

    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)

    # Log format
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

logger.info("=" * 70)
logger.info("DL CUSTOMER PREDICT - LOGGING STARTED")
logger.info("=" * 70)


2026-11-15 21:43:45 | INFO | ======================================================================
2026-11-15 21:43:45 | INFO | DL CUSTOMER PREDICT - LOGGING STARTED
2026-11-15 21:43:45 | INFO | ======================================================================


In [3]:
DATA_PATH = "./data/train_merged.csv"

train_df = pd.read_csv(DATA_PATH)

train_df["Date"] = pd.to_datetime(
    train_df["Date"]
)

print("\nCUSTOMER LSTM DATASET")
print("-" * 45)

print("Dataset shape:", train_df.shape)

print("Total stores:", train_df["Store"].nunique())

print("Date range:")
print(train_df["Date"].min(), "to", train_df["Date"].max())

print("\nRequired columns:")

print(
    train_df[
        ["Store", "Date", "Open", "Customers"]
    ].head()
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23284\2605391565.py:3: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(DATA_PATH)



CUSTOMER LSTM DATASET
---------------------------------------------
Dataset shape: (1017209, 24)
Total stores: 1115
Date range:
2013-01-01 00:00:00 to 2015-07-31 00:00:00

Required columns:
   Store       Date  Open  Customers
0      1 2015-07-31     1        555
1      2 2015-07-31     1        625
2      3 2015-07-31     1        821
3      4 2015-07-31     1       1498
4      5 2015-07-31     1        559


## 👥 LSTM Dataset Preparation

Prepare the historical customer count dataset for developing an LSTM-based Deep Learning model to forecast the number of customers visiting each store.

The `Customers` column will serve as the target variable, while `Store`, `Date`, and `Open` will provide the necessary store-level and temporal information.

| Parameter | Value |
|---|---:|
| Total Records | **1,017,209** |
| Total Columns | 24 |
| Unique Stores | 1,115 |
| Start Date | 2013-01-01 |
| End Date | 2015-07-31 |
| Target Variable | `Customers` |

### 👨‍🦰 Required Columns

| Column | Description | Role |
|---|---|---|
| `Store` | Unique store identifier | Grouping |
| `Date` | Daily observation date | Time Index |
| `Open` | Store operational status | Supporting Feature |
| `Customers` | Daily number of customers | Target Variable |

### 🔑 Key Findings

- The dataset contains approximately **1.02 million historical observations** across 1,115 stores.
- Historical customer information covers the period from January 2013 to July 2015.
- Each record represents a store's daily customer count.
- The `Customers` column will be used as the prediction target.
- Store-wise historical sequences will be maintained to preserve individual customer visitation patterns.

### Final Conclusion

The customer count dataset has been identified and the required columns are available for LSTM-based forecasting.

The next stage involves preparing store-level daily customer time series, examining customer visitation patterns, and transforming the historical observations into supervised learning sequences.

In [4]:
logger.info("=" * 70)
logger.info("CUSTOMER LSTM DATASET")
logger.info("=" * 70)

logger.info("Loading customer LSTM training dataset from: %s", DATA_PATH)

logger.info("Customer LSTM training dataset loaded successfully.")
logger.info("Dataset shape: %s", train_df.shape)

logger.info("Converting Date column into datetime format.")
logger.info("Date column converted successfully.")

logger.info("Total number of records: %d", len(train_df))

logger.info(
    "Total unique stores: %d",
    train_df["Store"].nunique()
)

logger.info(
    "Dataset date range: %s to %s",
    train_df["Date"].min(),
    train_df["Date"].max()
)

logger.info("Inspecting required columns: Store, Date, Open, Customers")

logger.info(
    "Required columns preview:\n%s",
    train_df[
        ["Store", "Date", "Open", "Customers"]
    ].head().to_string(index=False)
)

2026-11-15 21:56:22 | INFO | ======================================================================
2026-11-15 21:56:22 | INFO | CUSTOMER LSTM DATASET
2026-11-15 21:56:22 | INFO | ======================================================================
2026-11-15 21:56:22 | INFO | Loading customer LSTM training dataset from: ./data/train_merged.csv
2026-11-15 21:56:22 | INFO | Customer LSTM training dataset loaded successfully.
2026-11-15 21:56:22 | INFO | Dataset shape: (1017209, 24)
2026-11-15 21:56:22 | INFO | Converting Date column into datetime format.
2026-11-15 21:56:22 | INFO | Date column converted successfully.
2026-11-15 21:56:22 | INFO | Total number of records: 1017209
2026-11-15 21:56:23 | INFO | Total unique stores: 1115
2026-11-15 21:56:23 | INFO | Dataset date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00
2026-11-15 21:56:23 | INFO | Inspecting required columns: Store, Date, Open, Customers
2026-11-15 21:56:23 | INFO | Required columns preview:
 Store       Date  Ope

In [5]:
print("\nCUSTOMER DATA CHECK")
print("-" * 45)

print(
    "Missing Customers:",
    train_df["Customers"].isnull().sum()
)

print(
    "Negative Customers:",
    (train_df["Customers"] < 0).sum()
)

print(
    "Closed store records:",
    (train_df["Open"] == 0).sum()
)

print(
    "Closed stores with non-zero customers:",
    (
        (train_df["Open"] == 0) &
        (train_df["Customers"] != 0)
    ).sum()
)

print("\nCustomer statistics:")

print(
    train_df["Customers"].describe()
)


CUSTOMER DATA CHECK
---------------------------------------------
Missing Customers: 0
Negative Customers: 0
Closed store records: 172817
Closed stores with non-zero customers: 0

Customer statistics:
count    1.017209e+06
mean     6.331459e+02
std      4.644117e+02
min      0.000000e+00
25%      4.050000e+02
50%      6.090000e+02
75%      8.370000e+02
max      7.388000e+03
Name: Customers, dtype: float64


### 🤵‍♂️ Customer Count Data Quality Assessment

- **Complete Data:** No missing values were found in the `Customers` column.
- **Valid Values:** No negative customer counts were detected.
- **Closed Store Consistency:** All 172,817 closed-store records have zero customers, indicating consistent customer data for store closures.
- **Customer Distribution:** The average customer count is approximately 633, while the median is 609, indicating a slightly higher mean than median.
- **Variability:** The standard deviation of 464.41 indicates considerable variation in customer traffic across stores and dates.

#### Conclusion

The `Customers` variable passes the initial data quality checks, with no missing values, negative counts, or inconsistencies in closed-store records. The dataset is suitable for further customer count time-series analysis and LSTM model preparation. However, the wide variation in customer counts should be considered during model development and evaluation.

In [6]:
logger.info("=" * 70)
logger.info("CUSTOMER DATA VALIDATION")
logger.info("=" * 70)

logger.info(
    "Missing Customers: %d",
    train_df["Customers"].isnull().sum()
)

logger.info(
    "Negative Customers: %d",
    (train_df["Customers"] < 0).sum()
)

logger.info(
    "Closed store records: %d",
    (train_df["Open"] == 0).sum()
)

logger.info(
    "Closed stores with non-zero customers: %d",
    (
        (train_df["Open"] == 0) &
        (train_df["Customers"] != 0)
    ).sum()
)

logger.info(
    "Customer statistics:\n%s",
    train_df["Customers"].describe().to_string()
)

2026-11-15 22:03:03 | INFO | ======================================================================
2026-11-15 22:03:03 | INFO | CUSTOMER DATA VALIDATION
2026-11-15 22:03:03 | INFO | ======================================================================
2026-11-15 22:03:03 | INFO | Missing Customers: 0
2026-11-15 22:03:03 | INFO | Negative Customers: 0
2026-11-15 22:03:03 | INFO | Closed store records: 172817
2026-11-15 22:03:03 | INFO | Closed stores with non-zero customers: 0
2026-11-15 22:03:03 | INFO | Customer statistics:
count    1.017209e+06
mean     6.331459e+02
std      4.644117e+02
min      0.000000e+00
25%      4.050000e+02
50%      6.090000e+02
75%      8.370000e+02
max      7.388000e+03


In [7]:
gap_details = []

for store_id, store_data in train_df.groupby("Store"):

    store_data = store_data.sort_values("Date")

    dates = store_data["Date"].to_numpy()

    date_differences = (
        np.diff(dates)
        .astype("timedelta64[D]")
        .astype(int)
    )

    gap_positions = np.where(date_differences > 1)[0]

    for position in gap_positions:

        previous_date = pd.Timestamp(dates[position])
        next_date = pd.Timestamp(dates[position + 1])

        missing_days = (
            next_date - previous_date
        ).days - 1

        gap_details.append({
            "Store": store_id,
            "PreviousDate": previous_date,
            "NextDate": next_date,
            "MissingDays": missing_days
        })


gap_df = pd.DataFrame(gap_details)

print("\nMISSING DATE ANALYSIS")
print("-" * 50)

print("Total gaps:", len(gap_df))

if not gap_df.empty:

    print(
        "Total missing calendar days:",
        gap_df["MissingDays"].sum()
    )

    print(
        "Stores affected:",
        gap_df["Store"].nunique()
    )

    print("\nMissing days distribution:")
    print(gap_df["MissingDays"].value_counts().sort_index())

    print("\nStores with most gaps:")
    print(
        gap_df.groupby("Store")
        .agg(
            GapCount=("MissingDays", "count"),
            TotalMissingDays=("MissingDays", "sum")
        )
        .sort_values(
            "TotalMissingDays",
            ascending=False
        )
        .head(15)
    )

    print("\nFirst 20 gap records:")
    print(gap_df.head(20).to_string(index=False))

else:
    print("No missing dates found.")



MISSING DATE ANALYSIS
--------------------------------------------------
Total gaps: 180
Total missing calendar days: 33120
Stores affected: 180

Missing days distribution:
MissingDays
184    180
Name: count, dtype: int64

Stores with most gaps:
       GapCount  TotalMissingDays
Store                            
13            1               184
20            1               184
22            1               184
32            1               184
36            1               184
41            1               184
46            1               184
51            1               184
52            1               184
58            1               184
72            1               184
76            1               184
81            1               184
89            1               184
99            1               184

First 20 gap records:
 Store PreviousDate   NextDate  MissingDays
    13   2014-06-30 2015-01-01          184
    20   2014-06-30 2015-01-01          184
    22   2014-06-30 

### Missing Date Analysis – Customer Time Series

**Objective:** Identify missing calendar dates within each store's historical customer count records to assess time-series continuity before generating LSTM sequences.

#### 📆 Missing Date Overview

| Metric | Result |
|---|---:|
| Total Date Gaps | 180 |
| Total Missing Calendar Days | 33,120 |
| Stores Affected | 180 |
| Missing Days per Gap | 184 |
| Stores with Multiple Gaps | 0 |

#### 📆 Missing Days Distribution

| Missing Days | Number of Gaps |
|---:|---:|
| 184 | 180 |

Every affected store has exactly one gap of 184 calendar days.

#### 🔑 Key Findings

- **Consistent Gap Pattern:** All 180 affected stores have the same missing date interval, from July 1, 2014, through December 31, 2014.
- **Missing Period:** Each affected store is missing 184 consecutive calendar days.
- **Affected Stores:** 180 stores are impacted, while the remaining stores do not appear in this gap analysis.
- **Potential Data Limitation:** The common gap suggests a systematic interruption in the available historical records rather than randomly distributed missing dates.

#### ⚠️ Implications for LSTM Modelling

Missing calendar periods can affect sequence generation and temporal learning. A seven-record sliding window does not necessarily represent seven consecutive calendar days when date gaps exist.

For reliable LSTM training:
- Preserve chronological ordering within each store.
- Check date continuity before generating sequences.
- Avoid treating observations separated by 184 days as consecutive time steps.
- Do not automatically fill the missing period with zero customer counts, as missing observations do not necessarily indicate store closures.

#### Conclusion

The missing date analysis identifies a systematic 184-day interruption affecting 180 stores, resulting in 33,120 missing store-day observations. This issue must be considered during sequence preparation to prevent the LSTM model from learning misleading temporal relationships. A suitable approach is to exclude sequences that cross the missing interval or handle separate continuous periods independently.

In [8]:
logger.info("=" * 70)
logger.info("CUSTOMER LSTM MISSING DATE ANALYSIS")
logger.info("=" * 70)

logger.info("Checking missing calendar dates for each store.")

logger.info(
    "Total gaps detected: %d",
    len(gap_df)
)

if not gap_df.empty:

    logger.info(
        "Total missing calendar days: %d",
        gap_df["MissingDays"].sum()
    )

    logger.info(
        "Number of stores affected by missing dates: %d",
        gap_df["Store"].nunique()
    )

    logger.info(
        "Missing days distribution:\n%s",
        gap_df["MissingDays"].value_counts().sort_index().to_string()
    )

    logger.info(
        "Stores with most missing dates:\n%s",
        gap_df.groupby("Store")
        .agg(
            GapCount=("MissingDays", "count"),
            TotalMissingDays=("MissingDays", "sum")
        )
        .sort_values(
            "TotalMissingDays",
            ascending=False
        )
        .head(15)
        .to_string()
    )

    logger.info(
        "First 20 missing date gap records:\n%s",
        gap_df.head(20).to_string(index=False)
    )

else:
    logger.info("No missing calendar dates found across stores.")

2026-11-15 22:06:30 | INFO | ======================================================================
2026-11-15 22:06:30 | INFO | CUSTOMER LSTM MISSING DATE ANALYSIS
2026-11-15 22:06:30 | INFO | ======================================================================
2026-11-15 22:06:30 | INFO | Checking missing calendar dates for each store.
2026-11-15 22:06:30 | INFO | Total gaps detected: 180
2026-11-15 22:06:30 | INFO | Total missing calendar days: 33120
2026-11-15 22:06:30 | INFO | Number of stores affected by missing dates: 180
2026-11-15 22:06:30 | INFO | Missing days distribution:
MissingDays
184    180
2026-11-15 22:06:30 | INFO | Stores with most missing dates:
       GapCount  TotalMissingDays
Store                            
13            1               184
20            1               184
22            1               184
32            1               184
36            1               184
41            1               184
46            1               184
51            1  

In [9]:
WINDOW_SIZE = 7
VALIDATION_START = pd.Timestamp("2015-07-01")

print("\nCUSTOMER SEQUENCE PREPARATION")
print("-" * 50)

# Check duplicate Store-Date records
duplicate_count = train_df.duplicated(
    subset=["Store", "Date"]
).sum()

print("Duplicate Store-Date records:", duplicate_count)

if duplicate_count > 0:
    raise ValueError("Duplicate Store-Date records found!")

# Sort chronologically
train_df = train_df.sort_values(
    ["Store", "Date"]
).reset_index(drop=True)

# Sequence containers
X_train_list = []
y_train_list = []

X_valid_list = []
y_valid_list = []

total_segments = 0
segments_too_short = 0
stores_with_gaps = 0

# Process each store independently
for store_id, store_data in train_df.groupby("Store"):

    store_data = store_data.sort_values("Date")

    customers = store_data["Customers"].to_numpy(
        dtype=np.float32
    )

    dates = store_data["Date"].to_numpy()

    # Find locations where dates are not consecutive
    date_differences = (
        np.diff(dates)
        .astype("timedelta64[D]")
        .astype(int)
    )

    gap_positions = np.where(
        date_differences != 1
    )[0] + 1

    if len(gap_positions) > 0:
        stores_with_gaps += 1

    # Split store history into continuous segments
    segment_boundaries = np.concatenate([
        [0],
        gap_positions,
        [len(customers)]
    ])

    for i in range(len(segment_boundaries) - 1):

        start = segment_boundaries[i]
        end = segment_boundaries[i + 1]

        segment_customers = customers[start:end]
        segment_dates = dates[start:end]

        total_segments += 1

        # At least 8 days required:
        # 7 input days + 1 target day
        if len(segment_customers) <= WINDOW_SIZE:
            segments_too_short += 1
            continue

        # Create rolling windows
        windows = np.lib.stride_tricks.sliding_window_view(
            segment_customers,
            window_shape=WINDOW_SIZE + 1
        )

        X_store = windows[:, :-1]
        y_store = windows[:, -1]

        # Target date is the 8th day
        target_dates = segment_dates[WINDOW_SIZE:]

        # Chronological split
        train_mask = target_dates < np.datetime64(
            VALIDATION_START
        )

        valid_mask = target_dates >= np.datetime64(
            VALIDATION_START
        )

        if np.any(train_mask):
            X_train_list.append(
                X_store[train_mask]
            )
            y_train_list.append(
                y_store[train_mask]
            )

        if np.any(valid_mask):
            X_valid_list.append(
                X_store[valid_mask]
            )
            y_valid_list.append(
                y_store[valid_mask]
            )


# Combine all sequences
if not X_train_list or not X_valid_list:
    raise ValueError(
        "Training or validation sequences were not created."
    )

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

X_valid = np.concatenate(X_valid_list, axis=0)
y_valid = np.concatenate(y_valid_list, axis=0)


# Reshape for LSTM
X_train = X_train.reshape(-1, WINDOW_SIZE, 1)
X_valid = X_valid.reshape(-1, WINDOW_SIZE, 1)

y_train = y_train.reshape(-1, 1)
y_valid = y_valid.reshape(-1, 1)


# Display results
print("\nDATE GAP HANDLING")
print("-" * 50)

print("Stores containing date gaps:", stores_with_gaps)
print("Total continuous segments:", total_segments)
print("Segments too short for sequences:", segments_too_short)

print("\nSEQUENCE DATASET SUMMARY")
print("-" * 50)

print("Training input shape:", X_train.shape)
print("Training target shape:", y_train.shape)

print("Validation input shape:", X_valid.shape)
print("Validation target shape:", y_valid.shape)

print("\nTotal training sequences:", len(X_train))
print("Total validation sequences:", len(X_valid))

print("\nCustomer sequence preparation completed!")



CUSTOMER SEQUENCE PREPARATION
--------------------------------------------------
Duplicate Store-Date records: 0

DATE GAP HANDLING
--------------------------------------------------
Stores containing date gaps: 180
Total continuous segments: 1295
Segments too short for sequences: 0

SEQUENCE DATASET SUMMARY
--------------------------------------------------
Training input shape: (973579, 7, 1)
Training target shape: (973579, 1)
Validation input shape: (34565, 7, 1)
Validation target shape: (34565, 1)

Total training sequences: 973579
Total validation sequences: 34565

Customer sequence preparation completed!


### 🧍 Customer Count – LSTM Sequence Preparation

- No duplicate Store-Date records were detected.
- Date gaps were handled by splitting affected stores into continuous segments.
- All 1,295 segments were long enough to generate at least one valid sequence.
- The resulting dataset contains 973,579 training sequences and 34,565 validation sequences.
- The three-dimensional input shape `(samples, timesteps, features)` is suitable for LSTM model input, with 7 timesteps and 1 feature.

#### Conclusion

Customer sequence preparation was completed successfully. By generating sequences within continuous date segments, the dataset avoids incorrectly treating observations separated by missing calendar periods as consecutive days. The resulting **1,008,144 sequences** are ready for the next stage of LSTM preprocessing and model development.

In [10]:
logger.info("=" * 70)
logger.info("CUSTOMER LSTM SEQUENCE PREPARATION")
logger.info("=" * 70)

logger.info("Sequence window size: %d days", WINDOW_SIZE)
logger.info("Validation start date: %s", VALIDATION_START)

# Duplicate Store-Date validation
logger.info("Checking duplicate Store-Date records.")

logger.info(
    "Duplicate Store-Date records: %d",
    duplicate_count
)

# Chronological sorting
logger.info("Sorting customer dataset chronologically by Store and Date.")
logger.info("Customer dataset sorted successfully.")

# Sequence containers
logger.info("Initializing training and validation sequence containers.")

# Store-wise sequence processing
logger.info("Starting independent sequence preparation for each store.")
logger.info("Processing store-wise customer history.")
logger.info("Identifying non-consecutive dates and continuous segments.")

logger.info(
    "Stores containing date gaps: %d",
    stores_with_gaps
)

logger.info(
    "Total continuous segments identified: %d",
    total_segments
)

logger.info(
    "Segments too short for sequence generation: %d",
    segments_too_short
)

# Sliding window generation
logger.info(
    "Generating rolling customer sequences using %d-day windows.",
    WINDOW_SIZE
)

logger.info(
    "Sequence generation completed for all eligible continuous segments."
)

# Chronological split
logger.info(
    "Splitting sequences using validation start date: %s",
    VALIDATION_START
)

logger.info("Training sequences assigned to dates before validation start.")
logger.info("Validation sequences assigned to dates on or after validation start.")

# Sequence combination
logger.info("Combining store-wise training and validation sequences.")

logger.info("Training and validation sequences combined successfully.")

# LSTM reshaping
logger.info("Reshaping input sequences for LSTM compatibility.")
logger.info("Input shape format: (samples, timesteps, features).")
logger.info("Target shape format: (samples, 1).")

# Final sequence summary
logger.info(
    "Training input shape: %s",
    X_train.shape
)

logger.info(
    "Training target shape: %s",
    y_train.shape
)

logger.info(
    "Validation input shape: %s",
    X_valid.shape
)

logger.info(
    "Validation target shape: %s",
    y_valid.shape
)

logger.info(
    "Total training sequences: %d",
    len(X_train)
)

logger.info(
    "Total validation sequences: %d",
    len(X_valid)
)

2026-11-15 22:11:00 | INFO | ======================================================================
2026-11-15 22:11:00 | INFO | CUSTOMER LSTM SEQUENCE PREPARATION
2026-11-15 22:11:00 | INFO | ======================================================================
2026-11-15 22:11:00 | INFO | Sequence window size: 7 days
2026-11-15 22:11:00 | INFO | Validation start date: 2015-07-01 00:00:00
2026-11-15 22:11:00 | INFO | Checking duplicate Store-Date records.
2026-11-15 22:11:00 | INFO | Duplicate Store-Date records: 0
2026-11-15 22:11:00 | INFO | Sorting customer dataset chronologically by Store and Date.
2026-11-15 22:11:00 | INFO | Customer dataset sorted successfully.
2026-11-15 22:11:00 | INFO | Initializing training and validation sequence containers.
2026-11-15 22:11:00 | INFO | Starting independent sequence preparation for each store.
2026-11-15 22:11:00 | INFO | Processing store-wise customer history.
2026-11-15 22:11:00 | INFO | Identifying non-consecutive dates and continuous 

In [11]:

from sklearn.preprocessing import MinMaxScaler

print("\nCUSTOMER LSTM SCALING")
print("-" * 50)

# Create separate scalers
customer_X_scaler = MinMaxScaler(
    feature_range=(-1, 1)
)

customer_y_scaler = MinMaxScaler(
    feature_range=(-1, 1)
)


# ------------------------------------------
# 1. Fit input scaler using training inputs
# ------------------------------------------

# Flatten 3D input into 2D for scaling
X_train_2d = X_train.reshape(-1, 1)

# Fit only on training data
customer_X_scaler.fit(X_train_2d)

# Transform training and validation inputs
X_train_scaled = customer_X_scaler.transform(
    X_train_2d
).reshape(-1, WINDOW_SIZE, 1)

X_valid_scaled = customer_X_scaler.transform(
    X_valid.reshape(-1, 1)
).reshape(-1, WINDOW_SIZE, 1)


# ------------------------------------------
# 2. Fit target scaler using training targets
# ------------------------------------------

customer_y_scaler.fit(y_train)

y_train_scaled = customer_y_scaler.transform(
    y_train
)

y_valid_scaled = customer_y_scaler.transform(
    y_valid
)


# ------------------------------------------
# 3. Display scaling results
# ------------------------------------------

print("\nINPUT SCALING")
print("-" * 50)

print("X training shape:", X_train_scaled.shape)
print("X validation shape:", X_valid_scaled.shape)

print(
    "X training minimum:",
    X_train_scaled.min()
)

print(
    "X training maximum:",
    X_train_scaled.max()
)

print("\nTARGET SCALING")
print("-" * 50)

print("y training shape:", y_train_scaled.shape)
print("y validation shape:", y_valid_scaled.shape)

print(
    "y training minimum:",
    y_train_scaled.min()
)

print(
    "y training maximum:",
    y_train_scaled.max()
)

print("\nSCALER PARAMETERS")
print("-" * 50)

print(
    "Input scaler minimum:",
    customer_X_scaler.data_min_
)

print(
    "Input scaler maximum:",
    customer_X_scaler.data_max_
)

print(
    "Target scaler minimum:",
    customer_y_scaler.data_min_
)

print(
    "Target scaler maximum:",
    customer_y_scaler.data_max_
)

print("\nCustomer scaling completed successfully!")



CUSTOMER LSTM SCALING
--------------------------------------------------

INPUT SCALING
--------------------------------------------------
X training shape: (973579, 7, 1)
X validation shape: (34565, 7, 1)
X training minimum: -1.0
X training maximum: 0.9999999

TARGET SCALING
--------------------------------------------------
y training shape: (973579, 1)
y validation shape: (34565, 1)
y training minimum: -1.0
y training maximum: 0.9999999

SCALER PARAMETERS
--------------------------------------------------
Input scaler minimum: [0.]
Input scaler maximum: [7388.]
Target scaler minimum: [0.]
Target scaler maximum: [7388.]

Customer scaling completed successfully!


In [12]:
logger.info("=" * 70)
logger.info("CUSTOMER LSTM SCALING")
logger.info("=" * 70)

logger.info("Initializing separate MinMaxScaler objects for inputs and targets.")
logger.info("Scaling feature range: (-1, 1)")

# Input scaling
logger.info("Starting customer input scaling.")

logger.info(
    "Flattening training input from shape %s for scaler fitting.",
    X_train.shape
)

logger.info("Fitting input scaler using training data only.")

logger.info("Input scaler fitted successfully.")

logger.info("Transforming training and validation inputs.")

logger.info(
    "Scaled training input shape: %s",
    X_train_scaled.shape
)

logger.info(
    "Scaled validation input shape: %s",
    X_valid_scaled.shape
)

logger.info(
    "Scaled training input minimum: %f",
    X_train_scaled.min()
)

logger.info(
    "Scaled training input maximum: %f",
    X_train_scaled.max()
)

# Target scaling
logger.info("Starting customer target scaling.")

logger.info("Fitting target scaler using training targets only.")

logger.info("Target scaler fitted successfully.")

logger.info("Transforming training and validation targets.")

logger.info(
    "Scaled training target shape: %s",
    y_train_scaled.shape
)

logger.info(
    "Scaled validation target shape: %s",
    y_valid_scaled.shape
)

logger.info(
    "Scaled training target minimum: %f",
    y_train_scaled.min()
)

logger.info(
    "Scaled training target maximum: %f",
    y_train_scaled.max()
)

# Scaler parameters
logger.info(
    "Input scaler original minimum: %s",
    customer_X_scaler.data_min_
)

logger.info(
    "Input scaler original maximum: %s",
    customer_X_scaler.data_max_
)

logger.info(
    "Target scaler original minimum: %s",
    customer_y_scaler.data_min_
)

logger.info(
    "Target scaler original maximum: %s",
    customer_y_scaler.data_max_
)

2026-11-15 22:26:03 | INFO | ======================================================================
2026-11-15 22:26:03 | INFO | CUSTOMER LSTM SCALING
2026-11-15 22:26:03 | INFO | ======================================================================
2026-11-15 22:26:03 | INFO | Initializing separate MinMaxScaler objects for inputs and targets.
2026-11-15 22:26:03 | INFO | Scaling feature range: (-1, 1)
2026-11-15 22:26:03 | INFO | Starting customer input scaling.
2026-11-15 22:26:03 | INFO | Flattening training input from shape (973579, 7, 1) for scaler fitting.
2026-11-15 22:26:03 | INFO | Fitting input scaler using training data only.
2026-11-15 22:26:03 | INFO | Input scaler fitted successfully.
2026-11-15 22:26:03 | INFO | Transforming training and validation inputs.
2026-11-15 22:26:03 | INFO | Scaled training input shape: (973579, 7, 1)
2026-11-15 22:26:03 | INFO | Scaled validation input shape: (34565, 7, 1)
2026-11-15 22:26:03 | INFO | Scaled training input minimum: -1.000000


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

print("\nCUSTOMERS LSTM MODEL")
print("-" * 50)

# Build model
customer_lstm_model = Sequential([

    Input(shape=(WINDOW_SIZE, 1)),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    LSTM(
        32,
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(1)

])


# Compile model
customer_lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)


# Display model architecture
customer_lstm_model.summary()


# Early stopping
customer_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

print("\nCustomers LSTM model built successfully!")



CUSTOMERS LSTM MODEL
--------------------------------------------------


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 7, 64)          │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 7, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)


Customers LSTM model built successfully!


### 🛃 Customer Count – LSTM Model Architecture

**Objective:** Develop a stacked LSTM neural network to learn temporal patterns from the previous seven days of customer counts and predict the next day's customer traffic.

#### ♈ Model Architecture

The model consists of two LSTM layers, two Dropout layers, and one Dense output layer.

| Layer | Output Shape | Parameters | Purpose |
|---|---|---:|---|
| LSTM (64 units) | (None, 7, 64) | 16,896 | Extract temporal patterns from input sequences |
| Dropout | (None, 7, 64) | 0 | Reduce overfitting |
| LSTM (32 units) | (None, 32) | 12,416 | Learn higher-level sequential representations |
| Dropout | (None, 32) | 0 | Improve generalization |
| Dense (1 unit) | (None, 1) | 33 | Predict the next day's customer count |

```text
Input Sequence (7 Days, 1 Feature)
              |
              v
       LSTM (64 Units)
              |
              v
        Dropout Layer
              |
              v
       LSTM (32 Units)
              |
              v
        Dropout Layer
              |
              v
        Dense (1 Unit)
              |
              v
 Predicted Customer Count
```

#### 🗝️ Key Findings

- The model uses a stacked LSTM architecture with 64 and 32 units to capture sequential customer-count patterns.
- Dropout layers are included to help reduce overfitting.
- The Dense output layer produces a single next-day customer count prediction.
- All 29,345 parameters are trainable.
- The model architecture was created successfully.

#### Conclusion

The Customer Count LSTM model has been successfully built with a total of 29,345 trainable parameters. Its seven-day input structure is designed to learn short-term customer traffic patterns and generate next-day predictions. The model is ready for compilation and training.

In [14]:
logger.info("=" * 70)
logger.info("CUSTOMER LSTM MODEL")
logger.info("=" * 70)

logger.info("Initializing Customers LSTM Sequential model.")

logger.info(
    "Input shape: (%d timesteps, 1 feature)",
    WINDOW_SIZE
)

logger.info("Building first LSTM layer with 64 units and return_sequences=True.")
logger.info("Adding first Dropout layer with rate 0.2.")

logger.info("Building second LSTM layer with 32 units and return_sequences=False.")
logger.info("Adding second Dropout layer with rate 0.2.")

logger.info("Adding Dense output layer with 1 unit.")

logger.info("Customers LSTM model architecture created successfully.")

# Model compilation
logger.info("Compiling Customers LSTM model.")

logger.info("Optimizer: Adam")
logger.info("Loss function: Mean Squared Error (MSE)")
logger.info("Evaluation metric: Mean Absolute Error (MAE)")

logger.info("Customers LSTM model compiled successfully.")

# Model architecture
logger.info("Generating Customers LSTM model architecture summary.")

logger.info(
    "Customers LSTM model summary:\n%s",
    "\n".join(
        line for line in customer_lstm_model.summary(
            print_fn=lambda x: None
        ) or []
    )
)

# Early stopping
logger.info("Initializing EarlyStopping callback.")
logger.info("Monitoring validation loss.")
logger.info("EarlyStopping patience: 3 epochs.")
logger.info("Best model weights will be restored.")

logger.info("EarlyStopping callback configured successfully.")

2026-11-15 22:39:56 | INFO | ======================================================================
2026-11-15 22:39:56 | INFO | CUSTOMER LSTM MODEL
2026-11-15 22:39:56 | INFO | ======================================================================
2026-11-15 22:39:56 | INFO | Initializing Customers LSTM Sequential model.
2026-11-15 22:39:56 | INFO | Input shape: (7 timesteps, 1 feature)
2026-11-15 22:39:56 | INFO | Building first LSTM layer with 64 units and return_sequences=True.
2026-11-15 22:39:56 | INFO | Adding first Dropout layer with rate 0.2.
2026-11-15 22:39:56 | INFO | Building second LSTM layer with 32 units and return_sequences=False.
2026-11-15 22:39:56 | INFO | Adding second Dropout layer with rate 0.2.
2026-11-15 22:39:56 | INFO | Adding Dense output layer with 1 unit.
2026-11-15 22:39:57 | INFO | Customers LSTM model architecture created successfully.
2026-11-15 22:39:57 | INFO | Compiling Customers LSTM model.
2026-11-15 22:39:57 | INFO | Optimizer: Adam
2026-11-15 22

2026-11-15 22:39:57 | INFO | Customers LSTM model summary:

2026-11-15 22:39:57 | INFO | Initializing EarlyStopping callback.
2026-11-15 22:39:57 | INFO | Monitoring validation loss.
2026-11-15 22:39:57 | INFO | EarlyStopping patience: 3 epochs.
2026-11-15 22:39:57 | INFO | Best model weights will be restored.
2026-11-15 22:39:57 | INFO | EarlyStopping callback configured successfully.


In [15]:
print("\nCUSTOMERS LSTM TRAINING")
print("-" * 50)

EPOCHS = 20
BATCH_SIZE = 512

customer_history = customer_lstm_model.fit(

    X_train_scaled,
    y_train_scaled,

    validation_data=(
        X_valid_scaled,
        y_valid_scaled
    ),

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,

    callbacks=[
        customer_early_stopping
    ],

    shuffle=True,
    verbose=1
)


print("\nCustomers LSTM training completed!")

print(
    "Epochs actually completed:",
    len(customer_history.history["loss"])
)

print(
    "Best validation loss:",
    min(customer_history.history["val_loss"])
)



CUSTOMERS LSTM TRAINING
--------------------------------------------------
Epoch 1/20
1902/1902 ━━━━━━━━━━━━━━━━━━━━ 99s 48ms/step - loss: 0.0149 - mae: 0.0894 - val_loss: 0.0029 - val_mae: 0.0425
Epoch 2/20
1902/1902 ━━━━━━━━━━━━━━━━━━━━ 93s 49ms/step - loss: 0.0070 - mae: 0.0609 - val_loss: 0.0017 - val_mae: 0.0319
Epoch 3/20
1902/1902 ━━━━━━━━━━━━━━━━━━━━ 81s 43ms/step - loss: 0.0048 - mae: 0.0459 - val_loss: 0.0011 - val_mae: 0.0251
Epoch 4/20
1902/1902 ━━━━━━━━━━━━━━━━━━━━ 100s 53ms/step - loss: 0.0044 - mae: 0.0415 - val_loss: 0.0011 - val_mae: 0.0250
Epoch 5/20
1902/1902 ━━━━━━━━━━━━━━━━━━━━ 98s 52ms/step - loss: 0.0043 - mae: 0.0405 - val_loss: 0.0012 - val_mae: 0.0254
Epoch 6/20
1902/1902 ━━━━━━━━━━━━━━━━━━━━ 107s 56ms/step - loss: 0.0041 - mae: 0.0394 - val_loss: 0.0012 - val_mae: 0.0262
Epoch 7/20
1902/1902 ━━━━━━━━━━━━━━━━━━━━ 89s 46ms/step - loss: 0.0040 - mae: 0.0383 - val_loss: 8.8945e-04 - val_mae: 0.0205
Epoch 8/20
1902/1902 ━━━━━━━━━━━━━━━━━━━━ 81s 42ms/step - loss: 

In [17]:
logger.info("=" * 70)
logger.info("CUSTOMER LSTM TRAINING")
logger.info("=" * 70)

logger.info("Initializing Customers LSTM training.")
logger.info("Training input shape: %s", X_train_scaled.shape)
logger.info("Training target shape: %s", y_train_scaled.shape)
logger.info("Validation input shape: %s", X_valid_scaled.shape)
logger.info("Validation target shape: %s", y_valid_scaled.shape)

logger.info("Maximum training epochs: %d", EPOCHS)
logger.info("Batch size: %d", BATCH_SIZE)
logger.info("Optimizer: Adam")
logger.info("Loss function: Mean Squared Error (MSE)")
logger.info("EarlyStopping callback enabled.")
logger.info("EarlyStopping patience: %d epochs", customer_early_stopping.patience)
logger.info("Training shuffle enabled: True")

logger.info("Starting Customers LSTM model training.")

logger.info("Customers LSTM training completed successfully.")
logger.info(
    "Number of epochs actually completed: %d",
    len(customer_history.history["loss"])
)
logger.info(
    "Best validation loss: %f",
    min(customer_history.history["val_loss"])
)
logger.info(
    "Final training loss: %f",
    customer_history.history["loss"][-1]
)
logger.info(
    "Final validation loss: %f",
    customer_history.history["val_loss"][-1]
)

2026-11-15 22:56:40 | INFO | ======================================================================
2026-11-15 22:56:40 | INFO | CUSTOMER LSTM TRAINING
2026-11-15 22:56:40 | INFO | ======================================================================
2026-11-15 22:56:40 | INFO | Initializing Customers LSTM training.
2026-11-15 22:56:40 | INFO | Training input shape: (973579, 7, 1)
2026-11-15 22:56:40 | INFO | Training target shape: (973579, 1)
2026-11-15 22:56:40 | INFO | Validation input shape: (34565, 7, 1)
2026-11-15 22:56:40 | INFO | Validation target shape: (34565, 1)
2026-11-15 22:56:40 | INFO | Maximum training epochs: 20
2026-11-15 22:56:40 | INFO | Batch size: 512
2026-11-15 22:56:40 | INFO | Optimizer: Adam
2026-11-15 22:56:40 | INFO | Loss function: Mean Squared Error (MSE)
2026-11-15 22:56:40 | INFO | EarlyStopping callback enabled.
2026-11-15 22:56:40 | INFO | EarlyStopping patience: 3 epochs
2026-11-15 22:56:40 | INFO | Training shuffle enabled: True
2026-11-15 22:56:40 

In [18]:
print("\nCUSTOMERS LSTM MODEL EVALUATION")
print("-" * 50)

# Generate scaled predictions
y_valid_pred_scaled = customer_lstm_model.predict(
    X_valid_scaled,
    batch_size=1024,
    verbose=1
)

# Convert predictions back to actual customer counts
y_valid_pred = customer_y_scaler.inverse_transform(
    y_valid_pred_scaled
).flatten()

# Convert actual validation targets to 1D
y_valid_actual = y_valid.flatten()

# Customer counts cannot be negative
y_valid_pred = np.maximum(y_valid_pred, 0)


# ------------------------------------------
# Calculate evaluation metrics
# ------------------------------------------

mae = mean_absolute_error(
    y_valid_actual,
    y_valid_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid_actual,
        y_valid_pred
    )
)

# Exclude zero actual values from RMSPE
nonzero_mask = y_valid_actual > 0

rmspe = np.sqrt(
    np.mean(
        (
            (y_valid_actual[nonzero_mask] -
             y_valid_pred[nonzero_mask])
            /
            y_valid_actual[nonzero_mask]
        ) ** 2
    )
) * 100

r2 = r2_score(
    y_valid_actual,
    y_valid_pred
)


# ------------------------------------------
# Display results
# ------------------------------------------

print("\nCUSTOMERS LSTM VALIDATION METRICS")
print("-" * 50)

print(f"MAE:   {mae:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"RMSPE: {rmspe:.2f}%")
print(f"R2:    {r2:.4f}")

print("\nVALIDATION PREDICTION CHECK")
print("-" * 50)

print("Actual customers:")
print(y_valid_actual[:10])

print("\nPredicted customers:")
print(np.round(y_valid_pred[:10], 2))

print("\nEvaluation completed successfully!")



CUSTOMERS LSTM MODEL EVALUATION
--------------------------------------------------
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step

CUSTOMERS LSTM VALIDATION METRICS
--------------------------------------------------
MAE:   75.61
RMSE:  110.15
RMSPE: 15.09%
R2:    0.9359

VALIDATION PREDICTION CHECK
--------------------------------------------------
Actual customers:
[562. 573. 538. 560.   0. 540. 485. 485. 480. 449.]

Predicted customers:
[498.76 510.49 542.26 459.07  11.28 631.06 598.03 550.87 525.96 540.05]

Evaluation completed successfully!


### 🧗‍♂️ Customer Count – LSTM Model Evaluation

**Objective:** Evaluate the predictive performance of the trained LSTM model on the validation dataset using regression metrics and a comparison of actual versus predicted customer counts.

| Evaluation Metric | Result |
|---|---:|
| Mean Absolute Error (MAE) | 75.43 |
| Root Mean Squared Error (RMSE) | 110.34 |
| Root Mean Squared Percentage Error (RMSPE) | 15.20% |
| R² Score | 0.9357 |

#### 🥨 Actual vs Predicted Customer Counts

The following table compares the first 10 actual customer counts with the corresponding LSTM predictions.

| Observation | Actual Customers | Predicted Customers |
|---:|---:|---:|
| 1 | 562 | 453.11 |
| 2 | 573 | 472.50 |
| 3 | 538 | 521.59 |
| 4 | 560 | 443.81 |
| 5 | 0 | 0.00 |
| 6 | 540 | 607.81 |
| 7 | 485 | 575.88 |
| 8 | 485 | 503.36 |
| 9 | 480 | 495.37 |
| 10 | 449 | 524.24 |

#### 🔑 Key Findings

- **MAE of 75.43:** Predictions differ from actual customer counts by approximately 75 customers on average.
- **RMSE of 110.34:** Larger prediction errors have a greater influence on this metric, indicating some observations have relatively higher errors.
- **RMSPE of 15.20%:** The model achieves a 15.20% root mean squared percentage error under the evaluation procedure used.
- **R² of 0.9357:** The model explains approximately 93.57% of the variance in validation customer counts.
- **Closed Store Prediction:** The model correctly predicts zero customers for the fifth observation in the displayed sample.

#### Conclusion

The Customer Count LSTM model demonstrates strong validation performance, achieving an R² score of 0.9357 with an MAE of 75.43 customers. It captures a substantial proportion of the variation in customer traffic, although some individual predictions show noticeable deviations from actual values.

Overall, the results indicate that the model has learned useful temporal patterns in customer counts. Further evaluation across open-store observations and the final test dataset will help assess its practical forecasting performance.

*Note: RMSPE interpretation depends on how zero actual customer counts are handled. Confirm that zero-target observations are excluded or otherwise treated appropriately in the percentage-error calculation.*

In [19]:
logger.info("=" * 70)
logger.info("CUSTOMER LSTM MODEL EVALUATION")
logger.info("=" * 70)

logger.info("Generating scaled predictions for validation data.")
logger.info("Validation input shape: %s", X_valid_scaled.shape)

logger.info("Converting scaled predictions back to actual customer counts.")
logger.info("Predictions inverse-transformed successfully.")

logger.info("Converting actual validation targets to one-dimensional format.")
logger.info("Applying non-negative constraint to predicted customer counts.")

logger.info("Calculating Customer LSTM evaluation metrics.")

logger.info("Calculating Mean Absolute Error (MAE).")
logger.info("MAE: %.2f", mae)

logger.info("Calculating Root Mean Squared Error (RMSE).")
logger.info("RMSE: %.2f", rmse)

logger.info("Calculating Root Mean Squared Percentage Error (RMSPE).")
logger.info("Actual zero-customer records excluded from RMSPE calculation.")
logger.info("Non-zero actual validation records: %d", nonzero_mask.sum())
logger.info("RMSPE: %.2f%%", rmspe)

logger.info("Calculating R-squared (R2) score.")
logger.info("R2 Score: %.4f", r2)

logger.info("CUSTOMERS LSTM VALIDATION METRICS")
logger.info("-" * 50)
logger.info("MAE: %.2f", mae)
logger.info("RMSE: %.2f", rmse)
logger.info("RMSPE: %.2f%%", rmspe)
logger.info("R2: %.4f", r2)

logger.info("Validation prediction check.")
logger.info("Total validation records: %d", len(y_valid_actual))
logger.info("Actual customer counts (first 10): %s", y_valid_actual[:10])
logger.info(
    "Predicted customer counts (first 10): %s",
    np.round(y_valid_pred[:10], 2)
)

logger.info("Actual customer count range: %.2f to %.2f",
            y_valid_actual.min(), y_valid_actual.max())
logger.info("Predicted customer count range: %.2f to %.2f",
            y_valid_pred.min(), y_valid_pred.max())

2026-11-15 22:57:03 | INFO | ======================================================================
2026-11-15 22:57:03 | INFO | CUSTOMER LSTM MODEL EVALUATION
2026-11-15 22:57:03 | INFO | ======================================================================
2026-11-15 22:57:03 | INFO | Generating scaled predictions for validation data.
2026-11-15 22:57:03 | INFO | Validation input shape: (34565, 7, 1)
2026-11-15 22:57:03 | INFO | Converting scaled predictions back to actual customer counts.
2026-11-15 22:57:03 | INFO | Predictions inverse-transformed successfully.
2026-11-15 22:57:03 | INFO | Converting actual validation targets to one-dimensional format.
2026-11-15 22:57:03 | INFO | Applying non-negative constraint to predicted customer counts.
2026-11-15 22:57:03 | INFO | Calculating Customer LSTM evaluation metrics.
2026-11-15 22:57:03 | INFO | Calculating Mean Absolute Error (MAE).
2026-11-15 22:57:03 | INFO | MAE: 75.61
2026-11-15 22:57:03 | INFO | Calculating Root Mean Squared 

In [20]:
import os
from datetime import datetime
# ==========================================
# STEP 7: SAVE CUSTOMERS LSTM ARTIFACTS
# WITH CURRENT TIMESTAMP
# ==========================================

print("\nSAVING CUSTOMERS LSTM ARTIFACTS")
print("-" * 50)

# Create model directory
os.makedirs('deeplearning', exist_ok=True)

BASE_DIR = Path(
    r"D:\Digicrome AI\Internship\Nexthikes IT\Project6\Sales Forcast"
)

MODEL_DIR = BASE_DIR / "deeplearning"

# Generate current timestamp
timestamp = datetime.now().strftime(
    "%d-%m-%Y-%H-%M-%S-00"
)

# Generate timestamped file paths
CUSTOMER_MODEL_PATH = (
    MODEL_DIR / f"customer_lstm_{timestamp}.keras"
)

CUSTOMER_X_SCALER_PATH = (
    MODEL_DIR / f"customer_X_scaler_{timestamp}.pkl"
)

CUSTOMER_Y_SCALER_PATH = (
    MODEL_DIR / f"customer_y_scaler_{timestamp}.pkl"
)

CUSTOMER_CONFIG_PATH = (
    MODEL_DIR / f"customer_lstm_config_{timestamp}.json"
)


# ------------------------------------------
# 1. Serialize trained LSTM model
# ------------------------------------------

customer_lstm_model.save(
    CUSTOMER_MODEL_PATH
)


# ------------------------------------------
# 2. Serialize customer scalers
# ------------------------------------------

joblib.dump(
    customer_X_scaler,
    CUSTOMER_X_SCALER_PATH
)

joblib.dump(
    customer_y_scaler,
    CUSTOMER_Y_SCALER_PATH
)


# ------------------------------------------
# 3. Create model configuration
# ------------------------------------------

customer_config = {

    "model_name": "Customers LSTM",

    "target": "Customers",

    "input_features": ["Customers"],

    "window_size": WINDOW_SIZE,

    "input_shape": [WINDOW_SIZE, 1],

    "architecture": {
        "lstm_layer_1": 64,
        "lstm_layer_2": 32,
        "dropout": 0.2,
        "output_units": 1
    },

    "optimizer": "adam",

    "loss": "mse",

    "scaling": {
        "method": "MinMaxScaler",
        "feature_range": [-1, 1]
    },

    "validation_start": str(
        VALIDATION_START.date()
    ),

    "training_samples": int(len(X_train)),

    "validation_samples": int(len(X_valid)),

    "epochs_completed": len(
        customer_history.history["loss"]
    ),

    "best_validation_loss": float(
        min(customer_history.history["val_loss"])
    ),

    "metrics": {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "RMSPE": float(rmspe),
        "R2": float(r2)
    },

    "timestamp": timestamp
}


# ------------------------------------------
# 4. Serialize configuration
# ------------------------------------------

with open(
    CUSTOMER_CONFIG_PATH,
    "w"
) as file:

    json.dump(
        customer_config,
        file,
        indent=4
    )


# ------------------------------------------
# 5. Verify saved files
# ------------------------------------------

print("\nCUSTOMER MODEL SERIALIZATION COMPLETED")
print("-" * 50)

for file_path in [
    CUSTOMER_MODEL_PATH,
    CUSTOMER_X_SCALER_PATH,
    CUSTOMER_Y_SCALER_PATH,
    CUSTOMER_CONFIG_PATH
]:

    print(
        file_path.name,
        "->",
        "Saved" if file_path.exists() else "Missing"
    )

print("\nTimestamp:", timestamp)



SAVING CUSTOMERS LSTM ARTIFACTS
--------------------------------------------------

CUSTOMER MODEL SERIALIZATION COMPLETED
--------------------------------------------------
customer_lstm_15-11-2026-22-57-11-00.keras -> Saved
customer_X_scaler_15-11-2026-22-57-11-00.pkl -> Saved
customer_y_scaler_15-11-2026-22-57-11-00.pkl -> Saved
customer_lstm_config_15-11-2026-22-57-11-00.json -> Saved

Timestamp: 15-11-2026-22-57-11-00


In [21]:
logger.info("=" * 70)
logger.info("CUSTOMER LSTM ARTIFACT SAVING")
logger.info("=" * 70)

logger.info("Creating deep learning model directory.")
logger.info("Model directory: %s", MODEL_DIR)

logger.info("Generating timestamp for customer model artifacts.")
logger.info("Artifact timestamp: %s", timestamp)

logger.info("Preparing timestamped artifact file paths.")
logger.info("Customer LSTM model path: %s", CUSTOMER_MODEL_PATH)
logger.info("Customer input scaler path: %s", CUSTOMER_X_SCALER_PATH)
logger.info("Customer target scaler path: %s", CUSTOMER_Y_SCALER_PATH)
logger.info("Customer model configuration path: %s", CUSTOMER_CONFIG_PATH)

logger.info("Serializing trained Customers LSTM model.")
logger.info("Customers LSTM model saved successfully.")

logger.info("Serializing customer input scaler.")
logger.info("Customer input scaler saved successfully.")

logger.info("Serializing customer target scaler.")
logger.info("Customer target scaler saved successfully.")

logger.info("Creating Customers LSTM model configuration.")
logger.info("Model name: %s", customer_config["model_name"])
logger.info("Prediction target: %s", customer_config["target"])
logger.info("Input features: %s", customer_config["input_features"])
logger.info("Sequence window size: %d", customer_config["window_size"])
logger.info("Model input shape: %s", customer_config["input_shape"])
logger.info("LSTM architecture: %s", customer_config["architecture"])
logger.info("Optimizer: %s", customer_config["optimizer"])
logger.info("Loss function: %s", customer_config["loss"])
logger.info("Scaling configuration: %s", customer_config["scaling"])
logger.info("Validation start date: %s", customer_config["validation_start"])
logger.info("Training samples: %d", customer_config["training_samples"])
logger.info("Validation samples: %d", customer_config["validation_samples"])
logger.info("Epochs completed: %d", customer_config["epochs_completed"])
logger.info("Best validation loss: %f", customer_config["best_validation_loss"])
logger.info("Validation MAE: %.2f", customer_config["metrics"]["MAE"])
logger.info("Validation RMSE: %.2f", customer_config["metrics"]["RMSE"])
logger.info("Validation RMSPE: %.2f%%", customer_config["metrics"]["RMSPE"])
logger.info("Validation R2: %.4f", customer_config["metrics"]["R2"])

logger.info("Serializing Customers LSTM configuration to JSON.")
logger.info("Customer model configuration saved successfully.")

logger.info("Verifying saved customer model artifacts.")

for file_path in [
    CUSTOMER_MODEL_PATH,
    CUSTOMER_X_SCALER_PATH,
    CUSTOMER_Y_SCALER_PATH,
    CUSTOMER_CONFIG_PATH
]:
    if file_path.exists():
        logger.info("Artifact verified: %s -> Saved", file_path.name)
    else:
        logger.error("Artifact verification failed: %s -> Missing", file_path.name)

logger.info("Customer artifact timestamp: %s", timestamp)

2026-11-15 22:57:22 | INFO | ======================================================================
2026-11-15 22:57:22 | INFO | CUSTOMER LSTM ARTIFACT SAVING
2026-11-15 22:57:22 | INFO | ======================================================================
2026-11-15 22:57:22 | INFO | Creating deep learning model directory.
2026-11-15 22:57:22 | INFO | Model directory: D:\Digicrome AI\Internship\Nexthikes IT\Project6\Sales Forcast\deeplearning
2026-11-15 22:57:22 | INFO | Generating timestamp for customer model artifacts.
2026-11-15 22:57:22 | INFO | Artifact timestamp: 15-11-2026-22-57-11-00
2026-11-15 22:57:22 | INFO | Preparing timestamped artifact file paths.
2026-11-15 22:57:22 | INFO | Customer LSTM model path: D:\Digicrome AI\Internship\Nexthikes IT\Project6\Sales Forcast\deeplearning\customer_lstm_15-11-2026-22-57-11-00.keras
2026-11-15 22:57:22 | INFO | Customer input scaler path: D:\Digicrome AI\Internship\Nexthikes IT\Project6\Sales Forcast\deeplearning\customer_X_scaler_1

In [22]:
import joblib
import json

from tensorflow.keras.models import load_model

# Define artifact paths
CUSTOMER_MODEL_PATH = MODEL_DIR / "customer_lstm_14-11-2026-05-30-19-00.keras"

CUSTOMER_X_SCALER_PATH = MODEL_DIR / "customer_X_scaler_14-11-2026-05-30-19-00.pkl"

CUSTOMER_Y_SCALER_PATH = MODEL_DIR / "customer_y_scaler_14-11-2026-05-30-19-00.pkl"

CUSTOMER_CONFIG_PATH = MODEL_DIR / "customer_lstm_config_14-11-2026-05-30-19-00.json"


# Load saved model
loaded_customer_lstm = load_model(CUSTOMER_MODEL_PATH)

# Load saved scalers
loaded_customer_X_scaler = joblib.load(CUSTOMER_X_SCALER_PATH)

loaded_customer_y_scaler = joblib.load(CUSTOMER_Y_SCALER_PATH)

# Load model configuration
with open(CUSTOMER_CONFIG_PATH, "r") as file:
    loaded_customer_config = json.load(file)


# Display verification results
print("CUSTOMERS LSTM ARTIFACT VERIFICATION")
print("-" * 50)

print("Model loaded:", loaded_customer_lstm is not None)

print("Model input shape:", loaded_customer_lstm.input_shape)

print("Model output shape:", loaded_customer_lstm.output_shape)

print("X scaler loaded:", loaded_customer_X_scaler is not None)

print("Y scaler loaded:", loaded_customer_y_scaler is not None)

print("Configuration loaded:", loaded_customer_config is not None)

print("\nConfiguration:")
print(loaded_customer_config)

CUSTOMERS LSTM ARTIFACT VERIFICATION
--------------------------------------------------
Model loaded: True
Model input shape: (None, 7, 1)
Model output shape: (None, 1)
X scaler loaded: True
Y scaler loaded: True
Configuration loaded: True

Configuration:
{'model_name': 'Customers LSTM', 'target': 'Customers', 'input_features': ['Customers'], 'window_size': 7, 'input_shape': [7, 1], 'architecture': {'lstm_layer_1': 64, 'lstm_layer_2': 32, 'dropout': 0.2, 'output_units': 1}, 'optimizer': 'adam', 'loss': 'mse', 'scaling': {'method': 'MinMaxScaler', 'feature_range': [-1, 1]}, 'validation_start': '2015-07-01', 'training_samples': 973579, 'validation_samples': 34565, 'epochs_completed': 11, 'best_validation_loss': 0.0008932307828217745, 'metrics': {'MAE': 75.43267822265625, 'RMSE': 110.34274992466655, 'RMSPE': 15.199039459228516, 'R2': 0.9357154369354248}, 'timestamp': '14-11-2026-05-30-19-00'}


In [23]:
logger.info("=" * 70)
logger.info("CUSTOMER LSTM ARTIFACT LOADING")
logger.info("=" * 70)

logger.info("Defining saved Customers LSTM artifact paths.")
logger.info("Customer model path: %s", CUSTOMER_MODEL_PATH)
logger.info("Customer input scaler path: %s", CUSTOMER_X_SCALER_PATH)
logger.info("Customer target scaler path: %s", CUSTOMER_Y_SCALER_PATH)
logger.info("Customer configuration path: %s", CUSTOMER_CONFIG_PATH)

logger.info("Loading saved Customers LSTM model.")
logger.info("Customers LSTM model loaded successfully.")

logger.info("Loading saved customer input scaler.")
logger.info("Customer input scaler loaded successfully.")

logger.info("Loading saved customer target scaler.")
logger.info("Customer target scaler loaded successfully.")

logger.info("Loading Customers LSTM model configuration from JSON.")
logger.info("Customer model configuration loaded successfully.")

logger.info("=" * 70)
logger.info("CUSTOMERS LSTM ARTIFACT VERIFICATION")
logger.info("=" * 70)

logger.info("Model loaded: %s", loaded_customer_lstm is not None)
logger.info("Model input shape: %s", loaded_customer_lstm.input_shape)
logger.info("Model output shape: %s", loaded_customer_lstm.output_shape)

logger.info(
    "Input scaler loaded: %s",
    loaded_customer_X_scaler is not None
)
logger.info(
    "Target scaler loaded: %s",
    loaded_customer_y_scaler is not None
)
logger.info(
    "Configuration loaded: %s",
    loaded_customer_config is not None
)

logger.info("Loaded customer model configuration:")
logger.info(
    "Configuration details:\n%s",
    json.dumps(loaded_customer_config, indent=4)
)

2026-11-15 22:57:37 | INFO | ======================================================================
2026-11-15 22:57:37 | INFO | CUSTOMER LSTM ARTIFACT LOADING
2026-11-15 22:57:37 | INFO | ======================================================================
2026-11-15 22:57:37 | INFO | Defining saved Customers LSTM artifact paths.
2026-11-15 22:57:37 | INFO | Customer model path: D:\Digicrome AI\Internship\Nexthikes IT\Project6\Sales Forcast\deeplearning\customer_lstm_14-11-2026-05-30-19-00.keras
2026-11-15 22:57:37 | INFO | Customer input scaler path: D:\Digicrome AI\Internship\Nexthikes IT\Project6\Sales Forcast\deeplearning\customer_X_scaler_14-11-2026-05-30-19-00.pkl
2026-11-15 22:57:37 | INFO | Customer target scaler path: D:\Digicrome AI\Internship\Nexthikes IT\Project6\Sales Forcast\deeplearning\customer_y_scaler_14-11-2026-05-30-19-00.pkl
2026-11-15 22:57:37 | INFO | Customer configuration path: D:\Digicrome AI\Internship\Nexthikes IT\Project6\Sales Forcast\deeplearning\custo

In [ ]:

# Log completion before closing handlers
logger.info("=" * 70)
logger.info("Releasing logging resources")
logger.info("DL Model logging session closed successfully")
logger.info("=" * 70)

# Flush and close all handlers
for handler in logger.handlers[:]:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

# Release logger reference
logging.shutdown()

print("All logging resources released successfully.")